# Totales por circuito

Un JSON por (año, nivel) que unifica, por `circuito_id`, los totales de cada agrupación y de los "otros" (EN BLANCO, NULO, RECURRIDO, IMPUGNADO). Un archivo por nivel (no compartido entre niveles) para que no se pisen entre sí.

Output: `data/<año>/<nivel>/generales/circuito_<nivel>.json`, junto al `.json` (agregado) y `.csv` (oficial) de Generales, en su propia subcarpeta (`generales/`, hermana de `paso/` y `balotaje/` — §2.3):

```
{
  "anio": 2011,
  "nivel": "intendente",
  "categoria_id": 7,
  "fuente": "csv",
  "coincide_con_agregado_json": true,
  "advertencia_fuente": null,
  "cobertura": {
    "mesas_esperadas": 68,
    "mesas_totalizadas": 68,
    "mesas_totalizadas_porcentaje": 100.0,
    "cantidad_electores": 142345,
    "cantidad_votantes": 118932,
    "participacion_porcentaje": 83.55
  },
  "circuitos": {
    "460": {
      "mesas": 6,
      "electores": 2088,
      "mesas_sin_votos_positivos": 0,
      "positivos": {"0047": {"nombre": "...", "votos": 123, "campo_ideologico": "4"}, ...},
      "otros": {"EN BLANCO": 45, "NULO": 12, "RECURRIDO": 3, "IMPUGNADO": 1}
    },
    ...
  }
}
```

`campo_ideologico` se copia tal cual de `data/agrupaciones/agrupaciones.csv` / `agrupaciones_legislativas.csv`.

`circuito_id` se usa en su forma **canónica** (sin ceros a la izquierda — ver §1.2 más abajo), no como viene crudo en el CSV de cada año. `fuente`, `coincide_con_agregado_json`, `advertencia_fuente` y `cobertura` documentan de dónde sale el dato y qué tan completo está (§1.3/§1.4 del plan de correcciones). `mesas_sin_votos_positivos` señala mesas a revisar (padrón sin categoría vs. no escrutado), sin decidir la causa por sí solo.

In [1]:
import csv
import io
import json
import re
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))

from electoral.client import ResultadosClient
from electoral.models import EstadoRecuento

REPO = Path.cwd().parent
client = ResultadosClient(cache_dir=REPO / "data")

LA_PLATA = dict(
    tipo_eleccion=2,  # Generales
    distrito_id=2,  # Buenos Aires
    seccion_provincial_id=8,  # Sección Capital
    seccion_id=63,  # La Plata
)

## 1. Función de agregación por circuito

A partir del CSV oficial (ya en caché), agrupa por `circuito_id` **canónico** (ver §1.2) sumando `votos_cantidad`: por agrupación para las filas `POSITIVO`, y por `votos_tipo` para el resto. También cuenta, por circuito, cuántas mesas no tienen ningún voto positivo (`mesas_sin_votos_positivos`, §1.4) — una señal a revisar (padrón sin categoría vs. no escrutado), no una clasificación automática.

In [2]:
def normalizar_circuito_id(x):
    """Representación canónica de circuito_id: sin ceros a la izquierda.

    El mismo circuito aparece con ancho distinto según el año ("0460" en
    2011/2015, "000460" en 2019, "00460" en 2023) y algunos circuitos tienen
    sufijo de letra por subdivisión ("0496F"), que no es un problema de
    formato y hay que conservar. Se recorta solo la corrida de ceros líder
    antes del primer dígito no nulo, dejando cualquier sufijo intacto.
    """
    x = str(x).strip()
    m = re.match(r"^0+([0-9].*)$", x)
    return m.group(1) if m else x


CORRESPONDENCIAS_CIRCUITO = []


def agregar_por_circuito(df, anio, nivel):
    df = df.copy()
    df["circuito_id"] = df["circuito_id"].str.strip()  # el CSV trae algunos circuito_id con espacio de relleno
    df["agrupacion_nombre"] = df["agrupacion_nombre"].str.strip().str.upper()  # agrupaciones.csv usa nombres en mayúsculas (normalizado)
    df["circuito_id_canonico"] = df["circuito_id"].map(normalizar_circuito_id)

    for crudo, canonico in df[["circuito_id", "circuito_id_canonico"]].drop_duplicates().itertuples(index=False):
        CORRESPONDENCIAS_CIRCUITO.append(
            {"anio": anio, "nivel": nivel, "circuito_id_crudo": crudo, "circuito_id_canonico": canonico}
        )

    circuitos = {}
    for circuito_id, grupo in df.groupby("circuito_id_canonico"):
        positivos = {}
        for agrupacion_id, sub in grupo[grupo["votos_tipo"] == "POSITIVO"].groupby("agrupacion_id"):
            positivos[str(agrupacion_id)] = {
                "nombre": sub["agrupacion_nombre"].iloc[0],
                "votos": int(sub["votos_cantidad"].sum()),
            }

        otros = (
            grupo[grupo["votos_tipo"] != "POSITIVO"]
            .groupby("votos_tipo")["votos_cantidad"]
            .sum()
            .astype(int)
            .to_dict()
        )

        positivos_por_mesa = grupo[grupo["votos_tipo"] == "POSITIVO"].groupby("mesa_id")["votos_cantidad"].sum()
        mesas_sin_positivos = sum(
            1 for mesa_id in grupo["mesa_id"].unique() if positivos_por_mesa.get(mesa_id, 0) == 0
        )

        circuitos[circuito_id] = {
            "mesas": int(grupo["mesa_id"].nunique()),
            "electores": int(grupo.drop_duplicates("mesa_id")["mesa_electores"].sum()),
            "mesas_sin_votos_positivos": int(mesas_sin_positivos),
            "positivos": positivos,
            "otros": otros,
        }
    return circuitos

## 1.1 Campo ideológico

Cada agrupación en `positivos` suma el campo `campo_ideologico`, copiado tal cual está en `data/agrupaciones/agrupaciones.csv` / `agrupaciones_legislativas.csv`. El join es por (`anio`, `nivel`, `agrupacion`) exacto.

Si una agrupación no aparece en la clasificación, esto tiene que fallar (`KeyError`) en vez de guardar el circuito sin el campo — no hay valor "sin clasificar" implícito.

In [3]:
NIVEL_A_NIVEL_CSV = {"gobernador": "gobernacion"}


def cargar_clasificacion():
    filas = []
    for nombre in ["agrupaciones.csv", "agrupaciones_legislativas.csv"]:
        with open(REPO / "data" / "agrupaciones" / nombre, encoding="utf-8") as f:
            filas.extend(csv.DictReader(f))
    return {(r["anio"], r["nivel"], r["agrupacion"]): r["campo_ideologico"] for r in filas}


CLASIFICACION = cargar_clasificacion()


def agregar_campo_ideologico(circuitos, anio, nivel):
    nivel_csv = NIVEL_A_NIVEL_CSV.get(nivel, nivel)
    for c in circuitos.values():
        for info in c["positivos"].values():
            info["campo_ideologico"] = CLASIFICACION[(str(anio), nivel_csv, info["nombre"])]
    return circuitos

## 2. Caso de referencia: traer el CSV y aplicar la agregación (2011 / Intendente)

In [4]:
ANIO = 2011
NIVEL = "intendente"
CATEGORIA_ID = 7

csv_bytes = client.get_resultados_csv(
    anio_eleccion=ANIO, categoria_nombre=f"{NIVEL}/generales", categoria_id=CATEGORIA_ID, **LA_PLATA
)
df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)

circuitos = agregar_por_circuito(df, ANIO, NIVEL)
circuitos = agregar_campo_ideologico(circuitos, ANIO, NIVEL)
print(f"circuitos: {len(circuitos)}")
primero = next(iter(circuitos))
print(f"ejemplo ({primero}):", json.dumps(circuitos[primero], indent=2, ensure_ascii=False))

circuitos: 68
ejemplo (460): {
  "mesas": 6,
  "electores": 1987,
  "mesas_sin_votos_positivos": 0,
  "positivos": {
    "0047": {
      "nombre": "COALICIÓN CÍVICA - AFIRMACIÓN PARA UNA REPÚBLICA IGUALITARIA ARI",
      "votos": 131,
      "campo_ideologico": "4"
    },
    "0131": {
      "nombre": "ALIANZA FRENTE PARA LA VICTORIA",
      "votos": 430,
      "campo_ideologico": "3"
    },
    "0132": {
      "nombre": "ALIANZA FRENTE POPULAR",
      "votos": 113,
      "campo_ideologico": "3"
    },
    "0133": {
      "nombre": "ALIANZA COMPROMISO FEDERAL",
      "votos": 26,
      "campo_ideologico": "4"
    },
    "0134": {
      "nombre": "ALIANZA FRENTE AMPLIO PROGRESISTA",
      "votos": 295,
      "campo_ideologico": "2"
    },
    "0135": {
      "nombre": "ALIANZA FRENTE DE IZQUIERDA Y DE LOS TRABAJADORES",
      "votos": 54,
      "campo_ideologico": "1"
    },
    "0137": {
      "nombre": "ALIANZA UNIÓN PARA EL DESARROLLO SOCIAL",
      "votos": 152,
      "campo_ideologi

## 3. Validar contra el agregado de la sección

Sumar todos los circuitos tiene que dar exactamente el agregado de toda La Plata para esta categoría (ya sabemos, de los notebooks anteriores, que ese agregado es confiable salvo el caso conocido de Presidente 2019 — §1.5). De paso, se arma `cobertura` reutilizando los campos que la API ya expone en `estadoRecuento` (`EstadoRecuento`, `src/electoral/models.py`), sin recalcular nada por cuenta propia.

In [5]:
raw_agregado = client.get_resultados(
    anio_eleccion=ANIO, categoria_nombre=f"{NIVEL}/generales", categoria_id=CATEGORIA_ID, **LA_PLATA
)


def normalizar_agrupacion_id(x):
    x = str(x)
    return (x.lstrip("0") or "0") if x.isdigit() else x


positivos_circuitos = {}
otros_circuitos = {}
mesas_total = 0
electores_total = 0
for c in circuitos.values():
    mesas_total += c["mesas"]
    electores_total += c["electores"]
    for agrupacion_id, info in c["positivos"].items():
        clave = normalizar_agrupacion_id(agrupacion_id)
        positivos_circuitos[clave] = positivos_circuitos.get(clave, 0) + info["votos"]
    for tipo, votos in c["otros"].items():
        otros_circuitos[tipo] = otros_circuitos.get(tipo, 0) + votos

positivos_agregado = {
    normalizar_agrupacion_id(a["idAgrupacion"]): a["votos"] for a in raw_agregado["valoresTotalizadosPositivos"]
}

otros_total_circuitos = sum(otros_circuitos.values())
otros_total_agregado = (
    raw_agregado["valoresTotalizadosOtros"]["votosNulos"]
    + raw_agregado["valoresTotalizadosOtros"]["votosEnBlanco"]
    + raw_agregado["valoresTotalizadosOtros"]["votosRecurridosComandoImpugnados"]
)

estado_recuento = EstadoRecuento.from_json(raw_agregado["estadoRecuento"])
cobertura = {
    "mesas_esperadas": estado_recuento.mesas_esperadas,
    "mesas_totalizadas": estado_recuento.mesas_totalizadas,
    "mesas_totalizadas_porcentaje": estado_recuento.mesas_totalizadas_porcentaje,
    "cantidad_electores": estado_recuento.cantidad_electores,
    "cantidad_votantes": estado_recuento.cantidad_votantes,
    "participacion_porcentaje": estado_recuento.participacion_porcentaje,
}

coincide_con_agregado_json = (
    positivos_circuitos == positivos_agregado
    and otros_total_circuitos == otros_total_agregado
    and mesas_total == raw_agregado["estadoRecuento"]["mesasTotalizadas"]
    and electores_total == raw_agregado["estadoRecuento"]["cantidadElectores"]
)

print("categorías de 'otros' encontradas en este año:", sorted(otros_circuitos))
print("positivos: circuitos vs agregado ->",
      "OK" if positivos_circuitos == positivos_agregado else f"DIFERENCIA: {positivos_circuitos} vs {positivos_agregado}")
print("otros (total): circuitos =", otros_total_circuitos, "vs agregado =", otros_total_agregado)
print("mesas:", mesas_total, "vs", raw_agregado["estadoRecuento"]["mesasTotalizadas"])
print("electores:", electores_total, "vs", raw_agregado["estadoRecuento"]["cantidadElectores"])
print("cobertura:", cobertura)

assert coincide_con_agregado_json, "el caso de referencia (2011/intendente) tiene que coincidir exactamente"
print("\nOK: la suma por circuito coincide exactamente con el agregado de la sección.")

categorías de 'otros' encontradas en este año: ['EN BLANCO', 'IMPUGNADO', 'NULO', 'RECURRIDO']
positivos: circuitos vs agregado -> OK
otros (total): circuitos = 31989 vs agregado = 31989
mesas: 1429 vs 1429
electores: 493225 vs 493225
cobertura: {'mesas_esperadas': 0, 'mesas_totalizadas': 1429, 'mesas_totalizadas_porcentaje': 0, 'cantidad_electores': 493225, 'cantidad_votantes': 384273, 'participacion_porcentaje': 77.91}

OK: la suma por circuito coincide exactamente con el agregado de la sección.


## 4. Guardar `data/<año>/<nivel>/generales/circuito_<nivel>.json`

In [6]:
destino = REPO / "data" / str(ANIO) / NIVEL / "generales" / f"circuito_{NIVEL}.json"

contenido = {
    "anio": ANIO,
    "nivel": NIVEL,
    "categoria_id": CATEGORIA_ID,
    "fuente": "csv",
    "coincide_con_agregado_json": coincide_con_agregado_json,
    "advertencia_fuente": None if coincide_con_agregado_json else (
        "El JSON agregado crudo de esta consulta no coincide con el CSV oficial "
        "(ver README, sección de anomalías conocidas); se usó el CSV como única fuente confiable."
    ),
    "cobertura": cobertura,
    "circuitos": circuitos,
}

destino.write_text(json.dumps(contenido, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"{len(circuitos)} circuitos -> {destino}")

68 circuitos -> /workspaces/analisis-politica-economia/data/2011/intendente/generales/circuito_intendente.json


## 5. Función reutilizable + batch para todos los (año, nivel)

Empaqueta los pasos 2-4 en una función, con la misma validación pero sin `assert` (para que un caso con diferencia no frene los demás — el único caso conocido es Presidente 2019, §1.5). Al final también se vuelca `CORRESPONDENCIAS_CIRCUITO` (acumulada por `agregar_por_circuito` en cada llamada) a `data/agrupaciones/circuito_id_correspondencias.csv`, la tabla versionada de correspondencias entre el `circuito_id` crudo de cada año y el canónico (§1.2/§1.3).

In [7]:
def procesar(anio, nivel, categoria_id):
    csv_bytes = client.get_resultados_csv(
        anio_eleccion=anio, categoria_nombre=f"{nivel}/generales", categoria_id=categoria_id, **LA_PLATA
    )
    df = pd.read_csv(io.BytesIO(csv_bytes), low_memory=False)
    circuitos = agregar_por_circuito(df, anio, nivel)
    circuitos = agregar_campo_ideologico(circuitos, anio, nivel)

    raw_agregado = client.get_resultados(
        anio_eleccion=anio, categoria_nombre=f"{nivel}/generales", categoria_id=categoria_id, **LA_PLATA
    )

    positivos_circuitos, otros_circuitos = {}, {}
    mesas_total = electores_total = 0
    for c in circuitos.values():
        mesas_total += c["mesas"]
        electores_total += c["electores"]
        for agrupacion_id, info in c["positivos"].items():
            clave = normalizar_agrupacion_id(agrupacion_id)
            positivos_circuitos[clave] = positivos_circuitos.get(clave, 0) + info["votos"]
        for tipo, votos in c["otros"].items():
            otros_circuitos[tipo] = otros_circuitos.get(tipo, 0) + votos

    positivos_agregado = {
        normalizar_agrupacion_id(a["idAgrupacion"]): a["votos"] for a in raw_agregado["valoresTotalizadosPositivos"]
    }
    # "otros" se compara por total, no por nombre de categoría: votos_tipo no es estable
    # entre años (NULO/NULOS, EN BLANCO/BLANCOS...) y desde 2019 aparece COMANDO, que no
    # existía en 2011. agregar_por_circuito ya agrupa por lo que sea que diga votos_tipo,
    # así que el total siempre está bien igual — solo la comparación por nombre fallaría.
    otros_total_circuitos = sum(otros_circuitos.values())
    otros_total_agregado = (
        raw_agregado["valoresTotalizadosOtros"]["votosNulos"]
        + raw_agregado["valoresTotalizadosOtros"]["votosEnBlanco"]
        + raw_agregado["valoresTotalizadosOtros"]["votosRecurridosComandoImpugnados"]
    )

    coincide_con_agregado_json = (
        positivos_circuitos == positivos_agregado
        and otros_total_circuitos == otros_total_agregado
        and mesas_total == raw_agregado["estadoRecuento"]["mesasTotalizadas"]
        and electores_total == raw_agregado["estadoRecuento"]["cantidadElectores"]
    )

    estado_recuento = EstadoRecuento.from_json(raw_agregado["estadoRecuento"])
    cobertura = {
        "mesas_esperadas": estado_recuento.mesas_esperadas,
        "mesas_totalizadas": estado_recuento.mesas_totalizadas,
        "mesas_totalizadas_porcentaje": estado_recuento.mesas_totalizadas_porcentaje,
        "cantidad_electores": estado_recuento.cantidad_electores,
        "cantidad_votantes": estado_recuento.cantidad_votantes,
        "participacion_porcentaje": estado_recuento.participacion_porcentaje,
    }

    destino = REPO / "data" / str(anio) / nivel / "generales" / f"circuito_{nivel}.json"
    contenido = {
        "anio": anio,
        "nivel": nivel,
        "categoria_id": categoria_id,
        "fuente": "csv",
        "coincide_con_agregado_json": coincide_con_agregado_json,
        "advertencia_fuente": None if coincide_con_agregado_json else (
            "El JSON agregado crudo de esta consulta no coincide con el CSV oficial "
            "(ver README, sección de anomalías conocidas); se usó el CSV como única fuente confiable."
        ),
        "cobertura": cobertura,
        "circuitos": circuitos,
    }
    destino.write_text(json.dumps(contenido, indent=2, ensure_ascii=False), encoding="utf-8")

    return {
        "anio": anio,
        "nivel": nivel,
        "categoria_id": categoria_id,
        "ok": coincide_con_agregado_json,
        "circuitos": len(circuitos),
        "destino": str(destino),
        "otros_categorias": sorted(otros_circuitos),
    }

In [ ]:
EXECUTIVOS = [
    (anio, nivel, categoria_id)
    for anio in [2011, 2015, 2019, 2023]
    for nivel, categoria_id in [("presidente", 1), ("gobernador", 4), ("intendente", 7)]
]

# nacional usa Diputados Nacionales (idCargo=3
LEGISLATIVOS = [
    (2013, "nacional", 3), (2013, "provincial", 6), (2013, "municipal", 10),
    (2017, "nacional", 3), (2017, "provincial", 6), (2017, "municipal", 10),
    (2021, "nacional", 3), (2021, "provincial", 6), (2021, "municipal", 10),
    (2025, "nacional", 3),  # provincial/municipal no disponibles en 2025
]

resultados = []
for anio, nivel, categoria_id in EXECUTIVOS + LEGISLATIVOS:
    r = procesar(anio, nivel, categoria_id)
    resultados.append(r)
    estado = "OK" if r["ok"] else "DIFERENCIA vs agregado JSON"
    print(f"{anio}/{nivel} (idCargo={categoria_id}): {r['circuitos']} circuitos, {estado}")

no_ok = [r for r in resultados if not r["ok"]]
print(f"\n{len(resultados)} archivos generados. {len(no_ok)} con diferencia contra el agregado JSON "
      f"(esperado si ese agregado ya era conocido como no confiable):")
for r in no_ok:
    print(f"  {r['anio']}/{r['nivel']}")

# Tabla de correspondencias circuito_id crudo 
correspondencias = (
    pd.DataFrame(CORRESPONDENCIAS_CIRCUITO)
    .drop_duplicates()
    .sort_values(["anio", "nivel", "circuito_id_canonico"])
)
destino_correspondencias = REPO / "data" / "agrupaciones" / "circuito_id_correspondencias.csv"
correspondencias.to_csv(destino_correspondencias, index=False)
print(f"\n{len(correspondencias)} filas -> {destino_correspondencias}")

canonicos_por_anio = correspondencias.groupby("anio")["circuito_id_canonico"].apply(set)
comunes = set.intersection(*canonicos_por_anio)
todos = set.union(*canonicos_por_anio)
print(f"circuito_id canónicos comunes a todos los años procesados: {len(comunes)} de {len(todos)} en total")
print(f"variables entre años (revisar límites, no es solo formato): {sorted(todos - comunes)}")

2011/presidente (idCargo=1): 68 circuitos, OK


2011/gobernador (idCargo=4): 68 circuitos, OK


2011/intendente (idCargo=7): 68 circuitos, OK


2015/presidente (idCargo=1): 67 circuitos, OK


2015/gobernador (idCargo=4): 67 circuitos, OK


...

2017/municipal (idCargo=10): 69 circuitos, OK


2021/nacional (idCargo=3): 68 circuitos, OK


2021/provincial (idCargo=6): 69 circuitos, OK


2021/municipal (idCargo=10): 69 circuitos, OK


2025/nacional (idCargo=3): 68 circuitos, OK

22 archivos generados. 1 con diferencia contra el agregado JSON (esperado si ese agregado ya era conocido como no confiable):
  2019/presidente

1495 filas -> /workspaces/analisis-politica-economia/data/agrupaciones/circuito_id_correspondencias.csv
circuito_id canónicos comunes a todos los años procesados: 66 de 69 en total
variables entre años (revisar límites, no es solo formato): ['493', '496F', '504C']
